# 01 — PDF Text Extraction

This notebook is the **first stage of the data preparation pipeline** for the
Portuguese legal embedding fine-tuning project.

The source corpus contains legal documents in PDF format. Before we can identify
articles, create training examples, generate embeddings, or fine-tune a model,
we first need to convert those PDFs into a representation that Python can process
reliably.

The goal of this notebook is therefore deliberately simple:

> **Convert each PDF into structured, page-preserving JSON without trying to
> interpret the legal structure yet.**

## Where this notebook fits in the pipeline

```text
Legal PDFs
   ↓
01 — Text extraction        ← this notebook
   ↓
Page-preserving JSON
   ↓
02 — Legal structure parsing
   ↓
Articles and legal units
   ↓
Dataset construction
   ↓
Embedding baseline
   ↓
Hard-negative mining
   ↓
Fine-tuning
```

Keeping extraction and parsing as separate stages is important. If extraction,
cleaning, article detection and chunking were all performed at once, errors would
be much harder to identify and correct.

## Learning objectives

After completing this notebook, you should understand:

- how text can be extracted programmatically from a PDF;
- why page boundaries and source metadata should be preserved;
- why text cleaning should be conservative during extraction;
- how to create stable document identifiers;
- how to store extracted documents in JSON;
- how to perform basic quality checks before continuing the pipeline.

## What this notebook does

1. Finds all PDF files in `data/raw/`.
2. Extracts text from every page with PyMuPDF.
3. Applies minimal text normalisation.
4. Preserves the relationship between extracted text and its original page.
5. Stores metadata and simple quality indicators.
6. Saves one JSON file per legal document.
7. Produces a corpus-level extraction summary.

## What this notebook deliberately does not do

It does **not**:

- identify articles or paragraphs;
- remove recurring headers and footers;
- repair legal structure;
- create chunks;
- generate embeddings;
- perform OCR.

These tasks belong to later stages.

> **Important:** PyMuPDF extracts an existing text layer from the PDF. A scanned
> PDF may contain only images of pages and therefore return little or no text.
> The quality-control section helps identify such cases.

## 1. Imports

We begin by importing the Python libraries required by this stage.

### Main libraries

- `pathlib.Path` provides an operating-system-independent way of working with
  files and directories.
- `json` is used to save the extracted documents in a structured format.
- `re` provides regular expressions for small text-normalisation operations.
- `datetime` records when extraction was performed.
- `pymupdf` reads the PDF files and extracts their text.

### Why PyMuPDF?

A PDF is primarily a **page-description format**, not a text-document format.
What looks like a paragraph to a human may internally consist of many individual
text objects positioned at different coordinates.

PyMuPDF gives us access to the PDF text layer while retaining the concept of
individual pages. This is useful for legal documents because page provenance may
later be needed for citations, debugging and validation.

In [1]:
from pathlib import Path
import json
import re
from datetime import datetime, timezone

import pymupdf  # PyMuPDF


## 2. Project paths

A reproducible project should avoid hard-coded absolute paths such as:

```text
C:\Users\Name\Documents\project\data\raw
```

Such a path would normally work only on one computer.

Instead, this project defines all locations relative to the project root.

Expected structure:

```text
legal-embeddings-finetuning-pt/
├── data/
│   ├── raw/          ← original PDF files
│   └── extracted/    ← JSON files created by this notebook
└── notebooks/
    └── 01_extraction.ipynb
```

The code below supports running the notebook either from the project root or
from the `notebooks/` directory.

`mkdir(..., exist_ok=True)` creates the output directory when necessary without
raising an error if it already exists.

In [2]:
# The current working directory depends on how Jupyter was launched.
# If we are inside `notebooks/`, the project root is one directory above it.
# Otherwise, we assume the notebook was launched directly from the project root.
CWD = Path.cwd()
PROJECT_ROOT = CWD.parent if CWD.name == "notebooks" else CWD

# Input: original legal PDF documents.
RAW_DIR = PROJECT_ROOT / "data" / "raw"

# Output: page-preserving JSON documents created by this notebook.
EXTRACTED_DIR = PROJECT_ROOT / "data" / "extracted"

# `parents=True` creates missing parent directories if required.
# `exist_ok=True` means that re-running the notebook is safe if the folder exists.
EXTRACTED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root : {PROJECT_ROOT.resolve()}")
print(f"Raw PDFs     : {RAW_DIR.resolve()}")
print(f"Output       : {EXTRACTED_DIR.resolve()}")


Project root : C:\Users\user\Documents\GitHub\legal-embeddings-finetuning-pt
Raw PDFs     : C:\Users\user\Documents\GitHub\legal-embeddings-finetuning-pt\data\raw
Output       : C:\Users\user\Documents\GitHub\legal-embeddings-finetuning-pt\data\extracted


## 3. Discover source PDFs

Before processing the corpus, we need to know which source documents are
available.

`RAW_DIR.rglob("*")` searches recursively through `data/raw/`. We then retain
only files whose extension is `.pdf`.

Recursive discovery allows either of these organisations:

```text
data/raw/
├── civa.pdf
├── cirs.pdf
└── circ.pdf
```

or:

```text
data/raw/
├── tax/
│   ├── civa.pdf
│   ├── cirs.pdf
│   └── circ.pdf
└── budget/
    └── oe.pdf
```

The list is sorted so that processing order is deterministic. Re-running the
notebook with the same files therefore processes them in the same order.

### What to check in the output

Verify that:

- the expected number of PDFs was found;
- all intended legal documents appear;
- no unrelated PDF was accidentally placed in `data/raw/`.

In [3]:
# Search recursively for files below data/raw/ and keep only PDFs.
# `.suffix.lower()` also accepts filenames using an uppercase `.PDF` extension.
pdf_files = sorted(
    path for path in RAW_DIR.rglob("*")
    if path.is_file() and path.suffix.lower() == ".pdf"
)

print(f"PDF files found: {len(pdf_files)}")

# Display paths relative to the project root so the output remains readable
# and independent of the absolute path used on this computer.
for pdf_path in pdf_files:
    print(" -", pdf_path.relative_to(PROJECT_ROOT))

# Stop early if the source corpus is missing instead of silently producing
# an empty output directory.
if not pdf_files:
    raise FileNotFoundError(
        f"No PDF files were found in {RAW_DIR.resolve()}"
    )


PDF files found: 10
 - data\raw\CIMI.pdf
 - data\raw\CIRC.pdf
 - data\raw\CIRS.pdf
 - data\raw\CIS.pdf
 - data\raw\CIVA.pdf
 - data\raw\EBF.pdf
 - data\raw\LGT.pdf
 - data\raw\OE2026.pdf
 - data\raw\RGIT.pdf
 - data\raw\RITI.pdf


## 4. Minimal text normalisation

Text extracted from PDFs often contains small technical artefacts. Examples
include Windows-style line endings, null characters or trailing spaces.

We clean these artefacts, but **we deliberately avoid aggressive cleaning**.

### Why be conservative?

Suppose the PDF contains:

```text
Artigo 20.º
1 - São sujeitos passivos...
a) As pessoas singulares...
```

At this stage, line breaks may contain useful information about headings,
paragraphs and subparagraphs. Joining everything into one line could make the
next parsing stage more difficult.

The function below therefore performs only four operations:

1. standardises line endings to `\n`;
2. removes null characters (`\x00`);
3. removes spaces at the end of lines;
4. reduces unusually large runs of blank lines.

It does **not**:

- remove headers or footers;
- merge wrapped lines;
- repair hyphenated words;
- identify articles;
- change legal numbering.

This follows an important data-engineering principle:

> **Preserve information during early pipeline stages; transform it only when
> there is a clear reason to do so.**

In [4]:
def normalise_page_text(text: str) -> str:
    """Apply conservative normalisation to text extracted from one PDF page."""

    if not text:
        return ""

    # Normalise line endings and remove null characters.
    text = text.replace("\r\n", "\n").replace("\r", "\n").replace("\x00", "")

    # Remove trailing whitespace while preserving the original line structure.
    lines = [line.rstrip() for line in text.split("\n")]
    text = "\n".join(lines)

    # Avoid unusually large runs of blank lines.
    text = re.sub(r"\n{4,}", "\n\n\n", text)

    return text.strip()


## 5. Create a stable document identifier

Later stages need a reliable way to refer to each source document.

A filename is not always sufficient. For example, two directories could both
contain a file named `regulamento.pdf`.

The `make_document_id()` function derives an identifier from the PDF's path
relative to `data/raw/`.

For example:

```text
data/raw/tax/civa.pdf
```

can become:

```text
tax_civa
```

The identifier is:

- converted to lowercase;
- independent of Windows/Linux path separators;
- restricted to simple letters, numbers and underscores;
- suitable for later JSON records, datasets and vector databases.

### Why stable IDs matter

A typical retrieval pipeline will eventually contain relationships such as:

```text
document_id
    ↓
article_id
    ↓
passage_id
    ↓
embedding
```

If identifiers change unpredictably between executions, these relationships
become difficult to reproduce.

In [5]:
def make_document_id(pdf_path: Path, raw_dir: Path = RAW_DIR) -> str:
    """Create a filesystem-independent identifier for a source PDF."""

    relative_path = pdf_path.relative_to(raw_dir).with_suffix("")
    value = "_".join(relative_path.parts).lower()

    # Keep identifiers simple and suitable for later use in datasets/vector stores.
    value = re.sub(r"[^a-z0-9]+", "_", value)
    return value.strip("_")


## 6. Extract one PDF

We can now define the core extraction function.

The function opens one PDF and processes it **page by page**.

### Why preserve individual pages?

Instead of producing one very large string for the complete document, we store:

```text
Document
├── Page 1
├── Page 2
├── Page 3
└── ...
```

This gives us provenance. If a later parser identifies a problem in an article,
we can trace the text back to the page from which it originated.

Page boundaries are also useful for:

- generating citations;
- detecting recurring headers and footers;
- investigating extraction errors;
- comparing extracted text with the source PDF.

### `page.get_text("text", sort=True)`

`"text"` requests plain text extraction.

`sort=True` asks PyMuPDF to reconstruct the reading order using the positions of
text elements on the page. This is often preferable for legal documents with
headings, paragraphs and numbered provisions.

### Quality indicators

For every page we store:

- `character_count` — number of extracted characters;
- `word_count` — approximate number of whitespace-separated words;
- `is_empty` — whether no useful text was extracted.

These are not retrieval features. They are **data-quality diagnostics**.

### Document-level metadata

The final dictionary contains both the extracted pages and provenance such as
the source filename, source path, PDF metadata, extraction library and extraction
timestamp.

In [6]:
def extract_pdf(pdf_path: Path) -> dict:
    """Extract one PDF while preserving page-level provenance and metadata."""

    # Generate the identifier once so every downstream record can refer to
    # this document consistently.
    document_id = make_document_id(pdf_path)
    pages = []

    # The context manager closes the PDF automatically after extraction.
    with pymupdf.open(pdf_path) as pdf:
        metadata = pdf.metadata or {}

        # Human page numbering normally starts at 1, so enumerate(..., start=1)
        # is more convenient than Python's default zero-based indexing here.
        for page_number, page in enumerate(pdf, start=1):

            # `text` requests plain-text extraction.
            # `sort=True` uses page coordinates to improve reading order.
            raw_text = page.get_text("text", sort=True)

            # Apply only the conservative cleaning defined earlier.
            text = normalise_page_text(raw_text)

            # Keep the text together with simple diagnostics for this page.
            pages.append({
                "page": page_number,
                "text": text,
                "character_count": len(text),
                "word_count": len(text.split()) if text else 0,
                "is_empty": not bool(text.strip()),
            })

        # Build one structured record for the complete source document.
        document = {
            "document_id": document_id,
            "document_name": pdf_path.stem,
            "source_file": pdf_path.name,
            "source_path": pdf_path.relative_to(PROJECT_ROOT).as_posix(),
            "total_pages": len(pdf),
            "extraction": {
                "library": "PyMuPDF",
                "library_version": pymupdf.VersionBind,
                "extracted_at_utc": datetime.now(timezone.utc).isoformat(),
            },
            "pdf_metadata": {
                "title": metadata.get("title", ""),
                "author": metadata.get("author", ""),
                "subject": metadata.get("subject", ""),
                "keywords": metadata.get("keywords", ""),
            },
            "pages": pages,
        }

    return document


## 7. Save extracted documents as JSON

The extracted Python dictionary exists only in memory. We therefore persist each
document as a JSON file.

### Why JSON?

JSON is useful at this stage because it:

- preserves nested structures such as `document → pages`;
- is human-readable;
- is easy to inspect during debugging;
- can be loaded directly by Python;
- supports Unicode text.

A simplified output looks like:

```json
{
  "document_id": "civa",
  "document_name": "CIVA",
  "total_pages": 120,
  "pages": [
    {
      "page": 1,
      "text": "...",
      "character_count": 2450,
      "word_count": 390,
      "is_empty": false
    }
  ]
}
```

`ensure_ascii=False` is particularly important for Portuguese legal text. It
keeps characters such as `á`, `ç`, `ã` and `º` directly readable instead of
encoding them as Unicode escape sequences.

`indent=2` makes the files easier for humans to inspect.

In [7]:
def save_document(document: dict, output_dir: Path = EXTRACTED_DIR) -> Path:
    """Save an extracted document as formatted UTF-8 JSON."""

    output_path = output_dir / f"{document['document_id']}.json"

    with output_path.open("w", encoding="utf-8") as file:
        json.dump(document, file, ensure_ascii=False, indent=2)

    return output_path


## 8. Extract the complete corpus

Until now we have defined reusable functions. This cell actually executes the
pipeline over every PDF discovered earlier.

For each document:

```text
PDF
 ↓
extract_pdf()
 ↓
structured Python dictionary
 ↓
save_document()
 ↓
JSON file
```

We also calculate document-level statistics by aggregating the page-level
quality indicators.

### Error handling

Each PDF is processed inside a `try/except` block.

This is useful when processing a corpus because one malformed document should
not necessarily prevent all other documents from being extracted.

If extraction succeeds, `status` is set to `"ok"`.

If it fails, the exception is recorded in the summary with `status = "error"`.

### What to observe while this cell runs

The output should show one line per source PDF and the corresponding JSON file.
Any `ERROR` line should be investigated before continuing to the parsing stage.

In [8]:
extraction_summary = []

for index, pdf_path in enumerate(pdf_files, start=1):
    print(f"[{index}/{len(pdf_files)}] Extracting {pdf_path.name} ...")

    try:
        document = extract_pdf(pdf_path)
        output_path = save_document(document)

        empty_pages = sum(page["is_empty"] for page in document["pages"])
        total_characters = sum(page["character_count"] for page in document["pages"])
        total_words = sum(page["word_count"] for page in document["pages"])

        extraction_summary.append({
            "document_id": document["document_id"],
            "source_file": document["source_file"],
            "pages": document["total_pages"],
            "empty_pages": empty_pages,
            "characters": total_characters,
            "words": total_words,
            "output_file": output_path.name,
            "status": "ok",
            "error": None,
        })

        print(f"    Saved to {output_path.relative_to(PROJECT_ROOT)}")

    except Exception as exc:
        extraction_summary.append({
            "document_id": make_document_id(pdf_path),
            "source_file": pdf_path.name,
            "pages": None,
            "empty_pages": None,
            "characters": None,
            "words": None,
            "output_file": None,
            "status": "error",
            "error": str(exc),
        })

        print(f"    ERROR: {exc}")


[1/10] Extracting CIMI.pdf ...
    Saved to data\extracted\cimi.json
[2/10] Extracting CIRC.pdf ...
    Saved to data\extracted\circ.json
[3/10] Extracting CIRS.pdf ...
    Saved to data\extracted\cirs.json
[4/10] Extracting CIS.pdf ...
    Saved to data\extracted\cis.json
[5/10] Extracting CIVA.pdf ...
    Saved to data\extracted\civa.json
[6/10] Extracting EBF.pdf ...
    Saved to data\extracted\ebf.json
[7/10] Extracting LGT.pdf ...
    Saved to data\extracted\lgt.json
[8/10] Extracting OE2026.pdf ...
    Saved to data\extracted\oe2026.json
[9/10] Extracting RGIT.pdf ...
    Saved to data\extracted\rgit.json
[10/10] Extracting RITI.pdf ...
    Saved to data\extracted\riti.json


## 9. Inspect the extraction summary

Extraction success does not automatically mean extraction quality.

A PDF may technically open successfully while producing almost no useful text.
For example, a scanned legal document may contain hundreds of page images but no
embedded text layer.

The summary therefore reports:

- number of pages;
- number of empty pages;
- total extracted words;
- total extracted characters.

### How to interpret these values

A document with:

```text
200 pages | 0 empty | 120,000 words
```

is likely to contain a usable text layer.

A document with:

```text
200 pages | 190 empty | 800 words
```

deserves immediate investigation.

These statistics are only indicators. They do not prove that the extracted text
is correct, which is why we also perform manual inspection later.

In [9]:
# Display the extraction results without requiring pandas.
for item in extraction_summary:
    if item["status"] == "ok":
        print(
            f"{item['source_file']}: "
            f"{item['pages']} pages | "
            f"{item['empty_pages']} empty | "
            f"{item['words']:,} words | "
            f"{item['characters']:,} characters"
        )
    else:
        print(f"{item['source_file']}: ERROR — {item['error']}")


CIMI.pdf: 91 pages | 1 empty | 37,256 words | 282,714 characters
CIRC.pdf: 164 pages | 1 empty | 78,391 words | 548,561 characters
CIRS.pdf: 139 pages | 1 empty | 79,068 words | 550,025 characters
CIS.pdf: 64 pages | 1 empty | 25,009 words | 184,367 characters
CIVA.pdf: 188 pages | 1 empty | 83,049 words | 557,070 characters
EBF.pdf: 107 pages | 1 empty | 58,202 words | 390,769 characters
LGT.pdf: 65 pages | 1 empty | 28,116 words | 212,613 characters
OE2026.pdf: 270 pages | 0 empty | 75,738 words | 851,963 characters
RGIT.pdf: 73 pages | 1 empty | 27,247 words | 204,668 characters
RITI.pdf: 74 pages | 1 empty | 26,340 words | 176,332 characters


## 10. Save the corpus summary

The extraction summary is also saved as JSON.

This creates a machine-readable record of the extraction process that can later
be used to:

- compare corpus versions;
- document dataset statistics;
- identify problematic documents;
- support reproducibility.

The filename starts with `_` so that it is visually distinguishable from the
individual document JSON files.

In [10]:
summary_path = EXTRACTED_DIR / "_extraction_summary.json"

with summary_path.open("w", encoding="utf-8") as file:
    json.dump(extraction_summary, file, ensure_ascii=False, indent=2)

print(f"Summary saved to: {summary_path.relative_to(PROJECT_ROOT)}")


Summary saved to: data\extracted\_extraction_summary.json


## 11. Manually inspect a sample

Automated statistics cannot detect every extraction problem.

For example, a page may contain thousands of characters but still have:

- incorrect reading order;
- broken words;
- missing article numbers;
- duplicated text;
- corrupted Portuguese characters.

For this reason, we inspect actual extracted text before moving to parsing.

The code selects the first successfully extracted document and displays the
beginning of its first non-empty page.

### What should you look for?

Check whether the output preserves important legal elements such as:

```text
CAPÍTULO II

Artigo 3.º
Incidência

1 - ...

2 - ...

a) ...
b) ...
```

In particular, verify:

- accents and Portuguese characters;
- article numbering;
- paragraph numbering;
- subparagraph markers;
- headings;
- sensible reading order.

For a real production pipeline, it is advisable to inspect several pages from
different documents rather than relying on a single sample.

In [11]:
# Select the first successfully extracted document for a quick inspection.
successful_documents = [
    item for item in extraction_summary if item["status"] == "ok"
]

if successful_documents:
    sample_path = EXTRACTED_DIR / successful_documents[0]["output_file"]

    with sample_path.open("r", encoding="utf-8") as file:
        sample_document = json.load(file)

    print(f"Document: {sample_document['document_name']}")
    print(f"Pages: {sample_document['total_pages']}")
    print("-" * 80)

    # Show the beginning of the first non-empty page.
    sample_page = next(
        (page for page in sample_document["pages"] if not page["is_empty"]),
        None,
    )

    if sample_page:
        print(f"Page {sample_page['page']}")
        print(sample_page["text"][:3000])
    else:
        print("No page containing extractable text was found.")


Document: CIMI
Pages: 91
--------------------------------------------------------------------------------
Page 2
DECRETO-LEI Nº 287/2003, DE 12 DE NOVEMBRO (COM A ALTERAÇÃO PRODUZIDA PELA
LEI Nº60-A/2011, DE 30 DE NOVEMBRO) ............................................................................................ 7

CAPÍTULO III .......................................................................................................................................... 7

..REGIME TRANSITÓRIO ARTIGO 13.º ELEMENTOS PARA AVALIAÇÕES .............................. 7

  ARTIGO 14.º TAXAS DE CONSERVAÇÃO DE ESGOTOS ................................................................................ 7
  ARTIGO 15.º AVALIAÇÃO DE PRÉDIOS JÁ INSCRITOS NA MATRIZ............................................................... 7
  ARTIGO 15.º-A (ADITADO PELA LEI Nº 60-A/2011, DE 30 DE NOVEMBRO) PRINCÍPIOS DA AVALIAÇÃO
  GERAL .................................................................................

## 12. Automated quality checks

We now add two simple rules that flag suspicious documents.

### Rule 1 — More than 10% empty pages

A high proportion of empty pages may indicate:

- a scanned PDF without OCR;
- extraction problems;
- unusual page encoding.

### Rule 2 — Fewer than 250 characters per page on average

Legal documents normally contain substantial amounts of text. A very low
average may indicate that only headers, page numbers or fragments were
extracted.

### Important distinction: warning vs. error

These rules generate **warnings**, not automatic rejection.

A document may legitimately contain cover pages, tables, annexes or intentionally
blank pages. The purpose of a quality check is to direct human attention to
potential problems.

This illustrates a general principle for data preparation:

> Automated checks should help identify suspicious data, while final decisions
> should take the characteristics of the source documents into account.

In [12]:
for item in extraction_summary:
    if item["status"] != "ok":
        continue

    warnings = []

    if item["pages"] and item["empty_pages"] / item["pages"] > 0.10:
        warnings.append("more than 10% of pages are empty")

    if item["pages"] and item["characters"] / item["pages"] < 250:
        warnings.append("very low average character count per page")

    if warnings:
        print(f"WARNING — {item['source_file']}: " + "; ".join(warnings))

print("Quality check completed.")


Quality check completed.


## 13. What have we produced?

At the end of this notebook, the original PDF corpus has been transformed into a
page-preserving machine-readable representation:

```text
data/raw/
├── civa.pdf
├── cirs.pdf
└── ...

          ↓ extraction

data/extracted/
├── civa.json
├── cirs.json
├── ...
└── _extraction_summary.json
```

This is an important intermediate representation.

We have **not yet created an embedding dataset**. At this point we only have
faithfully extracted legal text plus provenance and quality metadata.

## Key concepts from this notebook

### Extraction vs. parsing

**Extraction** asks:

> What text can we obtain from the PDF?

**Parsing** asks:

> What does that text represent structurally?

Keeping these questions separate makes the pipeline easier to validate.

### Provenance

Every extracted text unit should remain traceable to its source. Here, page
numbers and source paths provide the first layer of provenance.

### Conservative preprocessing

Early preprocessing should avoid destroying information that later stages may
need.

### Quality control

A successful function call does not guarantee good data. Both automated
statistics and manual inspection are necessary.

---

## Next step — `02_parsing.ipynb`

The next notebook will interpret the extracted text as a legal document.

Conceptually, we want to move from:

```text
Page 1 → raw text
Page 2 → raw text
Page 3 → raw text
```

to something closer to:

```text
Document
└── Part / Title / Chapter / Section
    └── Article
        ├── Paragraph
        └── Subparagraph
```

The parsing stage can then:

- detect repeated page headers and footers;
- identify articles and their titles;
- reconstruct document hierarchy;
- preserve page provenance;
- assign stable identifiers to legal units.

That structured legal representation will later become the basis for dataset
construction and embedding fine-tuning.